# MOIA Zusatztests (ohne Zonen-/OD-Permutation)

Dieses Notebook setzt die zusaetzlichen Tests mit den MOIA-Daten um, **ohne** (3) Re-Zoning und **ohne** (4) OD-Permutation. Es erzeugt replizierbare Eingabedateien, Szenario-Varianten und Auswertungen.

## 1) Setup

Wir laden Bibliotheken, finden den Repo-Root und definieren Eingabe/Output-Pfade.

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)


def find_repo_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / "studies").is_dir():
            return parent
    return start

ROOT = find_repo_root(Path.cwd())
DATA_DIR = ROOT / "data" / "demand"
OUT_DIR = ROOT / "studies" / "moia_sensitivity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRIPS_FILE = DATA_DIR / "trips_klagenfurt.csv"
TRIPS_FILE

PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt.csv')

## 2) Nachfrage-Skalierung (±20-50%)

Wir erzeugen skalierte MOIA-Dateien durch Resampling. Die Dateien koennen spaeter in Szenarien referenziert werden.

In [11]:
scale_factors = [0.5, 0.8, 1.0, 1.2, 1.5]
trips = pd.read_csv(TRIPS_FILE)

scaled_paths = []
for f in scale_factors:
    target_n = max(1, int(round(len(trips) * f)))
    sample = trips.sample(n=target_n, replace=True, random_state=0).reset_index(drop=True)
    if "request_id" in sample.columns:
        sample["request_id"] = np.arange(1, len(sample) + 1)
    out_path = DATA_DIR / f"trips_klagenfurt_scaled_{str(f).replace('.', 'p')}.csv"
    sample.to_csv(out_path, index=False)
    scaled_paths.append(out_path)

scaled_paths

[PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt_scaled_0p5.csv'),
 PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt_scaled_0p8.csv'),
 PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt_scaled_1p0.csv'),
 PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt_scaled_1p2.csv'),
 PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt_scaled_1p5.csv')]

## 3) Zeitfenster-Slicing

Wir erzeugen Subsets fuer Morgen/Mittag/Abend (auf Basis der departure_time-Spalte).

In [12]:
def slice_by_time(df: pd.DataFrame, start_sec: int, end_sec: int) -> pd.DataFrame:
    if "departure_time" not in df.columns:
        raise ValueError("departure_time fehlt in trips_klagenfurt.csv")
    seconds = df["departure_time"] % 86400
    mask = (seconds >= start_sec) & (seconds < end_sec)
    return df[mask].copy()

windows = {
    "morning_7_9": (7 * 3600, 9 * 3600),
    "midday_12_14": (12 * 3600, 14 * 3600),
    "evening_17_19": (17 * 3600, 19 * 3600),
}

sliced_paths = []
for label, (s, e) in windows.items():
    subset = slice_by_time(trips, s, e)
    out_path = DATA_DIR / f"trips_klagenfurt_{label}.csv"
    subset.to_csv(out_path, index=False)
    sliced_paths.append(out_path)

sliced_paths

[PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt_morning_7_9.csv'),
 PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt_midday_12_14.csv'),
 PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/data/demand/trips_klagenfurt_evening_17_19.csv')]

## 4) Service-Constraints variieren (Wartezeit/Detour)

Wir erzeugen eine Szenario-CSV mit Varianten fuer max. Wartezeit und Detour-Faktor.

In [13]:
service_variants = pd.DataFrame([
    {"scenario_name": "svc_base", "op_max_wait_time": 480, "op_max_detour_time_factor": 40},
    {"scenario_name": "svc_wait_300", "op_max_wait_time": 300, "op_max_detour_time_factor": 40},
    {"scenario_name": "svc_wait_600", "op_max_wait_time": 600, "op_max_detour_time_factor": 40},
    {"scenario_name": "svc_detour_20", "op_max_wait_time": 480, "op_max_detour_time_factor": 20},
    {"scenario_name": "svc_detour_60", "op_max_wait_time": 480, "op_max_detour_time_factor": 60},
])

svc_path = OUT_DIR / "service_constraints.csv"
service_variants.to_csv(svc_path, index=False)
svc_path

PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/moia_sensitivity/service_constraints.csv')

## 5) UC3: Anteil Barrierefreiheit erhoehen

Wir erzeugen eine erweiterte UC3-Szenario-CSV mit hoeheren Anteilen.

In [14]:
uc3_extended = pd.DataFrame([
    {"scenario_name": "mixed_60", "op_fleet_composition": "standard_8seat:8;accessible_12seat:12", "user_max_wait_time": 180, "user_max_wait_time_2": 300},
    {"scenario_name": "mixed_80", "op_fleet_composition": "standard_8seat:4;accessible_12seat:16", "user_max_wait_time": 180, "user_max_wait_time_2": 300},
])

uc3_path = OUT_DIR / "uc3_scenarios_extended.csv"
uc3_extended.to_csv(uc3_path, index=False)
uc3_path

PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/moia_sensitivity/uc3_scenarios_extended.csv')

## 6) UC2: Interventionsstaerke erhoehen

Wir erzeugen eine aggressive Variante, um Schwellenwerte zu testen.

In [15]:
uc2_strong = pd.DataFrame([
    {"scenario_name": "phone_support_strong", "user_max_decision_time": 300, "user_max_wait_time": 240, "user_max_wait_time_2": 420, "op_const_boarding_time": 45},
    {"scenario_name": "simple_ui_strong", "user_max_decision_time": 180, "user_max_wait_time": 360, "user_max_wait_time_2": 540, "op_const_boarding_time": 45},
    {"scenario_name": "full_support_strong", "user_max_decision_time": 360, "user_max_wait_time": 420, "user_max_wait_time_2": 600, "op_const_boarding_time": 30},
])

uc2_path = OUT_DIR / "uc2_scenarios_strong.csv"
uc2_strong.to_csv(uc2_path, index=False)
uc2_path

PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/moia_sensitivity/uc2_scenarios_strong.csv')

## 7) Pooling-Kompatibilitaet analysieren

Wir schaetzen, wie viele Trips innerhalb enger Zeitfenster und raeumlich kompatibel sind.

In [16]:
def compatible_pairs(df: pd.DataFrame, time_window_sec: int = 300, max_dist_deg: float = 0.01) -> int:
    # sehr einfache Schaetzung: gleiche Zeitfenster + naheliegende O- und D-Punkte
    if not {"departure_time", "origin_x", "origin_y", "destination_x", "destination_y"}.issubset(df.columns):
        raise ValueError("Spalten fehlen fuer Kompatibilitaetsanalyse")
    df = df.copy()
    df["t_bin"] = (df["departure_time"] // time_window_sec).astype(int)
    pairs = 0
    for _, group in df.groupby("t_bin"):
        if len(group) < 2:
            continue
        coords_o = group[["origin_x", "origin_y"]].to_numpy()
        coords_d = group[["destination_x", "destination_y"]].to_numpy()
        # naive Distanz: L_inf auf Grad
        for i in range(len(group)):
            for j in range(i + 1, len(group)):
                if max(abs(coords_o[i] - coords_o[j])) <= max_dist_deg and max(abs(coords_d[i] - coords_d[j])) <= max_dist_deg:
                    pairs += 1
    return pairs

pair_estimate = compatible_pairs(trips, time_window_sec=300, max_dist_deg=0.01)
pair_estimate

0

## 8) Repositioning-Strategien aktivieren

Wir erstellen eine Konfigurationsdatei mit unterschiedlichen Repositioning-Methoden (nur als Vorlage, keine Ausfuehrung).

In [17]:
repo_methods = pd.DataFrame([
    {"scenario_name": "repo_none", "op_repo_method": "None"},
    {"scenario_name": "repo_basic", "op_repo_method": "Basic"},
    {"scenario_name": "repo_hist", "op_repo_method": "HistoricalDemand"},
])

repo_path = OUT_DIR / "repositioning_variants.csv"
repo_methods.to_csv(repo_path, index=False)
repo_path

PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/moia_sensitivity/repositioning_variants.csv')

## 9) Kosten-Robustheit (Auswertungsvorlage)

Wir erstellen eine Funktion, um Kosten pro bedienter Anfrage aus bestehenden Ergebnissen zu berechnen.

In [18]:
def cost_per_served(eval_path: Path) -> float | None:
    df = pd.read_csv(eval_path, index_col=0)
    col = df.columns[0]
    if "mod fix costs" not in df.index or "mod var costs" not in df.index or "number users" not in df.index:
        return None
    served = float(df.loc["number users", col])
    if served <= 0:
        return None
    total_cost = float(df.loc["mod fix costs", col]) + float(df.loc["mod var costs", col])
    return total_cost / served

# Beispiel: erste gefundene standard_eval.csv
example_eval = None
for p in (ROOT / "studies").rglob("standard_eval.csv"):
    example_eval = p
    break

example_eval, cost_per_served(example_eval) if example_eval else None

(PosixPath('/Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/uc3_study/results/mixed_50/standard_eval.csv'),
 1366.1333333333334)

## 10) Hinweise zur Ausfuehrung

- Skalierte und geschnittene Nachfrage-Dateien liegen unter `data/demand/`.
- Szenario-CSV-Dateien liegen unter `studies/moia_sensitivity/`.
- In den jeweiligen Studien laesst sich der Demand- oder Szenario-CSV-Pfad referenzieren, um die Varianten zu starten.

Hinweis: Dieses Notebook erzeugt nur die Dateien und Auswertungslogik, fuehrt jedoch keine Simulationen aus.

## 11) Zusammenfassung der erzeugten Outputs

Diese Zelle listet alle erzeugten Dateien und gibt eine kurze, lesbare Zusammenfassung. Damit ist klar, *was* dieses Notebook produziert hat.

In [20]:
from pathlib import Path

root = Path().resolve()
while not (root / "studies").exists():
    root = root.parent

scaling_files = sorted((root / "data" / "demand").glob("trips_klagenfurt_scaled_*.csv"))
time_slice_files = sorted((root / "data" / "demand").glob("trips_klagenfurt_*_*.csv"))

time_slice_files = [
    p
    for p in time_slice_files
    if p.name.startswith("trips_klagenfurt_")
    and any(tag in p.name for tag in ["morning", "midday", "evening"])
]

moia_study = root / "studies" / "moia_sensitivity"

scenario_files = {
    "service_constraints": moia_study / "service_constraints.csv",
    "uc2_strong": moia_study / "uc2_scenarios_strong.csv",
    "uc3_extended": moia_study / "uc3_scenarios_extended.csv",
    "repositioning_variants": moia_study / "repositioning_variants.csv",
}

print("Skalierte Trips-Dateien:")
for p in scaling_files:
    print(f"- {p.relative_to(root)}")

print("\nZeitfenster-Dateien:")
for p in time_slice_files:
    print(f"- {p.relative_to(root)}")

print("\nSzenario-CSV-Dateien:")
for name, p in scenario_files.items():
    status = "OK" if p.exists() else "FEHLT"
    print(f"- {name}: {p.relative_to(root)} [{status}]")

print("\nPooling-Kompatibilitaet (Schnellcheck):")
print("- Ergebnis im Notebook: 0 kompatible Paare (5 min, 0.01 Grad) -> sehr strenges Kriterium")
print("- Empfehlung: Falls gewuenscht, Schwellwerte fuer Test erweitern")

Skalierte Trips-Dateien:
- data/demand/trips_klagenfurt_scaled_0p5.csv
- data/demand/trips_klagenfurt_scaled_0p8.csv
- data/demand/trips_klagenfurt_scaled_1p0.csv
- data/demand/trips_klagenfurt_scaled_1p2.csv
- data/demand/trips_klagenfurt_scaled_1p5.csv

Zeitfenster-Dateien:
- data/demand/trips_klagenfurt_evening_17_19.csv
- data/demand/trips_klagenfurt_midday_12_14.csv
- data/demand/trips_klagenfurt_morning_7_9.csv

Szenario-CSV-Dateien:
- service_constraints: studies/moia_sensitivity/service_constraints.csv [OK]
- uc2_strong: studies/moia_sensitivity/uc2_scenarios_strong.csv [OK]
- uc3_extended: studies/moia_sensitivity/uc3_scenarios_extended.csv [OK]
- repositioning_variants: studies/moia_sensitivity/repositioning_variants.csv [OK]

Pooling-Kompatibilitaet (Schnellcheck):
- Ergebnis im Notebook: 0 kompatible Paare (5 min, 0.01 Grad) -> sehr strenges Kriterium
- Empfehlung: Falls gewuenscht, Schwellwerte fuer Test erweitern
